# Exploratory

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
from sklearn.linear_model import LinearRegression

from flamesafe import config
from flamesafe.analysis import stats
from flamesafe.physics.atmospheres import REFERENCE_ATMOSPHERES

import duckdb
from flamesafe import config

import math


In [ ]:

# Leer dataframe gold gas flame
con = duckdb.connect(str(config.data_dir() / "gold" / "flamesafe.duckdb"), read_only=True)

gas = con.sql("""
    SELECT g.*, i.acronym, i.tier, i.hardware_family, i.environments
    FROM gas_flame_tests g
    LEFT JOIN investigations i USING (psi_id)
""").df()      # sigue teniendo 660 filas: una por prueba


In [ ]:
pubs = con.sql("""
    SELECT i.psi_id, i.acronym, i.tier, p.doi, p.title
    FROM (SELECT *, unnest(psi_ids) AS psi_id FROM publications) p
    JOIN investigations i USING (psi_id)
""").df()

In [ ]:
gas.info()

In [ ]:
gas.head()

### Split por datos

In [ ]:
numerical_columns = gas.select_dtypes(include=['int64','float64']).columns
len(numerical_columns)
numerical_data = gas[numerical_columns]

In [ ]:
categorical_columns = gas.select_dtypes(include=["object","str"]).columns
categorical_columns
categorical_data = gas[categorical_columns]

In [ ]:
boolean_columns = gas.select_dtypes(include=['bool']).columns
boolean_columns
categorical_data = gas[boolean_columns]

### Numerical Analysis

In [ ]:
import matplotlib.pyplot as plt

numerical_data.hist(bins=50, figsize=(20, 15))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import math

ncols = 3
nrows = math.ceil(len(numerical_data.columns) / ncols)

fig, axes = plt.subplots(
    nrows, ncols, figsize=(20, 15), squeeze=False
)

for ax, col in zip(axes.flat, numerical_data.columns):
    values = numerical_data[col].dropna()

    if values.nunique() > 1:
        values.plot.kde(ax=ax, linewidth=2)
        ax.set_ylabel("Density")
    else:
        ax.text(0.5, 0.5, "Insufficient variation",
                ha="center", va="center", transform=ax.transAxes)

    ax.set_title(col)
    ax.set_xlabel(col)
    ax.grid(alpha=0.3)

for ax in axes.flat[len(numerical_data.columns):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.collections import EllipseCollection

# 1. Preparar las mediciones numéricas.
# Excluir identificadores y categorías codificadas con números.
data = (
    numerical_data
    .select_dtypes(include="number")
    .drop(columns=["test_day", "source_row", "tier"], errors="ignore")
    .replace([np.inf, -np.inf], np.nan)
)

# Excluir columnas constantes o sin observaciones.
data = data.loc[:, data.nunique() > 1]

# Cada correlación requiere al menos 10 pares de observaciones válidas.
corr = data.corr(method="pearson", min_periods=10)

# 2. Configurar la figura.
n = len(corr.columns)
fig, ax = plt.subplots(figsize=(max(10, n * 0.75), max(9, n * 0.7)))
ax.set_aspect("equal")

# 3. Preparar posiciones y correlaciones válidas.
x, y = np.meshgrid(np.arange(n), np.arange(n))
corr_flat = corr.to_numpy().ravel()
valid = np.isfinite(corr_flat)

r = corr_flat[valid]

# Una correlación fuerte produce una elipse más estrecha.
# El mínimo evita que las correlaciones ±1 desaparezcan.
widths = np.maximum(1 - np.abs(r), 0.025)
heights = np.ones_like(r)

# Con el eje Y invertido, estos ángulos muestran:
# positiva: /    negativa: \
angles = np.where(r >= 0, 45, -45)

# 4. Crear las elipses.
ec = EllipseCollection(
    widths=widths * 0.85,
    heights=heights * 0.85,
    angles=angles,
    units="xy",
    offsets=np.column_stack([x.ravel()[valid], y.ravel()[valid]]),
    transOffset=ax.transData,
    array=r,
    cmap=sns.diverging_palette(20, 220, as_cmap=True),
    clim=(-1, 1),
)

ax.add_collection(ec)

# 5. Etiquetas y límites.
ax.set_xlim(-0.5, n - 0.5)
ax.set_ylim(n - 0.5, -0.5)

ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(corr.columns, rotation=60, ha="right", fontsize=9)
ax.set_yticklabels(corr.columns, fontsize=9)

ax.set_xticks(np.arange(n + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(n + 1) - 0.5, minor=True)
ax.grid(which="minor", color="lightgray", linewidth=0.5)
ax.tick_params(which="minor", bottom=False, left=False)

colorbar = fig.colorbar(ec, ax=ax, shrink=0.8, pad=0.03)
colorbar.set_label("Correlación de Pearson")

ax.set_title(
    "Correlación entre variables numéricas\n"
    "Celdas vacías: datos insuficientes o correlación indefinida",
    pad=20,
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

MAX_FALTANTES_PCA = 0.30

EXCLUIR_PCA = [
    # Identificadores y categorías numéricas
    "test_day", "source_row", "tier",

    # Resultados posteriores al experimento
    "ignited", "ignite_eventdata", "extinction_mode",
    "burn_time_s", "cool_flame", "peak_burner_t_k", "tfp",
    "hc_burnrate_mg_s",

    # Variable derivada de xo2 y p_kpa
    "po2_kpa",

    # Condición final, no inicial
    "xo2_final",
]

# 1. Seleccionar variables y limpiar valores no finitos.
datos = (
    numerical_data
    .select_dtypes(include="number")
    .drop(columns=EXCLUIR_PCA, errors="ignore")
    .replace([np.inf, -np.inf], np.nan)
    .astype(float)
)

# 2. Conservar columnas con ≤30 % de faltantes y variación.
perfil_pca = pd.DataFrame({
    "pct_faltantes": datos.isna().mean() * 100,
    "n_unicos": datos.nunique(),
})

aptas = (
    perfil_pca["pct_faltantes"].le(100 * MAX_FALTANTES_PCA)
    & perfil_pca["n_unicos"].gt(1)
)

datos_pca = datos.loc[:, aptas]

print("Variables incluidas:", datos_pca.columns.tolist())
display(perfil_pca.assign(incluida=aptas).round(2))

if datos_pca.shape[1] >= 2 and len(datos_pca) >= 3:
    # 3. Imputar con la mediana y estandarizar.
    faltantes_pca = datos_pca.isna().sum()
    x_pca = datos_pca.fillna(datos_pca.median())

    escalador = StandardScaler()
    x_estandarizado = escalador.fit_transform(x_pca)

    # 4. Ajustar PCA.
    modelo_pca = PCA().fit(x_estandarizado)

    varianza = modelo_pca.explained_variance_ratio_
    acumulada = varianza.cumsum()
    componentes = np.arange(1, len(varianza) + 1)

    varianza_pca = pd.DataFrame({
        "componente": componentes,
        "varianza_pct": 100 * varianza,
        "acumulada_pct": 100 * acumulada,
    })

    # Coeficientes de cada variable en los componentes.
    pesos_pca = pd.DataFrame(
        modelo_pca.components_.T,
        index=x_pca.columns,
        columns=[f"PC{k}" for k in componentes],
    )

    # Coordenadas de cada prueba en el espacio PCA.
    scores_pca = pd.DataFrame(
        modelo_pca.transform(x_estandarizado),
        index=x_pca.index,
        columns=pesos_pca.columns,
    )

    # 5. Visualizar varianza explicada.
    fig, (ax_var, ax_acum) = plt.subplots(
        1, 2, figsize=(13, 4.5), layout="constrained"
    )

    ax_var.bar(componentes, 100 * varianza, color="steelblue")
    ax_var.set(
        title="Varianza por componente",
        xlabel="Componente",
        ylabel="Varianza explicada (%)",
        xticks=componentes,
    )

    ax_acum.plot(
        componentes, 100 * acumulada,
        marker="o", color="tab:orange"
    )
    ax_acum.set(
        title="Varianza acumulada",
        xlabel="Número de componentes",
        ylabel="Varianza explicada (%)",
        ylim=(0, 105),
        xticks=componentes,
    )

    for umbral in (0.80, 0.90):
        k = min(
            int(np.searchsorted(acumulada, umbral)) + 1,
            len(componentes),
        )
        ax_acum.axhline(
            100 * umbral, ls="--", lw=1,
            color="gray", label=f"{umbral:.0%}"
        )
        ax_acum.scatter(
            k, 100 * acumulada[k - 1],
            color="tab:orange", zorder=3
        )
        print(f"{umbral:.0%} de varianza explicada: {k} componentes")

    ax_acum.legend()
    plt.show()

    display(varianza_pca.round(2))

    # 6. Variables con mayor contribución a los primeros componentes.
    for componente in pesos_pca.columns[:3]:
        pesos = pesos_pca[componente]
        principales = pesos.abs().nlargest(5).index

        print(f"\n{componente}: mayores pesos absolutos")
        display(pesos.loc[principales].round(3).to_frame("peso"))

    if faltantes_pca.any():
        print("\nValores imputados con la mediana:")
        display(
            faltantes_pca[faltantes_pca > 0].to_frame("n_imputados")
        )

else:
    print(
        "Se necesitan al menos dos variables con variación "
        "y tres filas para este análisis."
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

UMBRAL = 0.40
MIN_PARES = 20

# 1. Preparar mediciones numéricas.
datos = (
    numerical_data
    .select_dtypes(include="number")
    .drop(columns=["test_day", "source_row", "tier"], errors="ignore")
    .replace([np.inf, -np.inf], np.nan)
    .astype(float)
)

datos = datos.loc[:, datos.nunique() > 1]

# 2. Pearson con observaciones completas para cada par.
corr = datos.corr(method="pearson", min_periods=MIN_PARES)

i, j = np.triu_indices(len(corr.columns), k=1)
r = corr.to_numpy()[i, j]

seleccion = np.isfinite(r) & (np.abs(r) > UMBRAL)

pares = pd.DataFrame({
    "variable_1": corr.columns[i[seleccion]],
    "variable_2": corr.columns[j[seleccion]],
    "r": r[seleccion],
})

pares["n"] = [
    len(datos[[a, b]].dropna())
    for a, b in zip(pares["variable_1"], pares["variable_2"])
]

pares = (
    pares.assign(abs_r=pares["r"].abs())
    .sort_values("abs_r", ascending=False)
    .reset_index(drop=True)
)

n_pares = len(pares)

if n_pares == 0:
    print(
        f"No se encontraron pares con |r| > {UMBRAL:.2f} "
        f"y al menos {MIN_PARES} observaciones."
    )

else:
    n_cols = min(3, n_pares)
    n_filas = int(np.ceil(n_pares / n_cols))

    fig, axes = plt.subplots(
        n_filas, n_cols,
        figsize=(5.5 * n_cols, 4.2 * n_filas),
        squeeze=False,
        layout="constrained",
    )

    for ax, fila in zip(axes.flat, pares.itertuples(index=False)):
        var1, var2 = fila.variable_1, fila.variable_2
        par = datos[[var1, var2]].dropna()
        x, y = par.to_numpy().T

        # KDE necesita variación en ambas variables y datos
        # que no estén exactamente sobre una recta.
        puede_kde = (
            par.nunique().gt(1).all()
            and np.linalg.matrix_rank(
                par.to_numpy() - par.to_numpy().mean(axis=0)
            ) == 2
        )

        metodo = "hexbin"

        if puede_kde:
            try:
                sns.kdeplot(
                    x=x,
                    y=y,
                    ax=ax,
                    fill=True,
                    cmap="Blues",
                    gridsize=60,
                    levels=8,
                    cut=0,
                    warn_singular=False,
                )

                if ax.collections:
                    metodo = "KDE"

            except (ValueError, np.linalg.LinAlgError):
                ax.clear()

        if metodo == "hexbin":
            ax.hexbin(
                x, y,
                gridsize=30,
                mincnt=1,
                bins="log",
                cmap="Blues",
            )

        # Mostrar también las observaciones reales.
        ax.scatter(x, y, s=9, alpha=0.25, color="black")

        ax.set(
            xlabel=var1,
            ylabel=var2,
            title=f"r = {fila.r:.2f} | n = {fila.n} | {metodo}",
        )

    for ax in axes.flat[n_pares:]:
        ax.set_visible(False)

    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

UMBRAL = 0.40
MIN_PARES = 20

# 1. Preparar variables numéricas.
datos = (
    numerical_data
    .select_dtypes(include="number")
    .drop(columns=["test_day", "source_row", "tier"], errors="ignore")
    .replace([np.inf, -np.inf], np.nan)
    .astype(float)
)

datos = datos.loc[:, datos.nunique() > 1]

# 2. Seleccionar pares únicos por correlación absoluta.
corr = datos.corr(method="pearson", min_periods=MIN_PARES)

i, j = np.triu_indices(len(corr.columns), k=1)
r = corr.to_numpy()[i, j]

seleccion = np.isfinite(r) & (np.abs(r) > UMBRAL)

pares = pd.DataFrame({
    "variable_1": corr.columns[i[seleccion]],
    "variable_2": corr.columns[j[seleccion]],
    "r": r[seleccion],
})

pares = (
    pares.assign(abs_r=pares["r"].abs())
    .sort_values("abs_r", ascending=False)
    .reset_index(drop=True)
)

n_pares = len(pares)

if n_pares == 0:
    print(
        f"No se encontraron pares con |r| > {UMBRAL:.2f} "
        f"y al menos {MIN_PARES} observaciones."
    )

else:
    n_cols = min(3, n_pares)
    n_filas = int(np.ceil(n_pares / n_cols))

    fig, axes = plt.subplots(
        n_filas, n_cols,
        figsize=(5.5 * n_cols, 4.2 * n_filas),
        squeeze=False,
        layout="constrained",
    )

    for ax, fila in zip(axes.flat, pares.itertuples(index=False)):
        variables = [fila.variable_1, fila.variable_2]

        # Las dos distribuciones usan exactamente las mismas pruebas.
        par = datos[variables].dropna()

        # Estandarización dentro de las observaciones válidas del par.
        escala = par.std(ddof=0).replace(0, 1)
        estandarizado = (par - par.mean()) / escala

        sns.violinplot(
            data=estandarizado,
            inner="quartile",
            cut=0,
            gridsize=60,
            color="steelblue",
            ax=ax,
        )

        ax.axhline(0, color="gray", linestyle="--", linewidth=1)
        ax.tick_params(axis="x", labelrotation=20)
        ax.set(
            title=f"Pearson r = {fila.r:.2f} | n = {len(par)}",
            ylabel="Puntaje z",
            xlabel="",
        )

    for ax in axes.flat[n_pares:]:
        ax.set_visible(False)

    plt.show()

<!-- FLAMESAFE:SEGMENTATION:BEGIN -->
## Revisión del EDA y metodología de segmentación experimental

**Fecha de revisión: 2026-11-15. Unidad de observación: una prueba experimental,
identificada por `psi_id + test_id`.** Las celdas anteriores se conservan como
exploración global; esta sección establece qué comparaciones son interpretables.

### Qué se ha explorado y cuáles son sus límites

Hasta aquí se cargaron 660 pruebas mediante DuckDB y se revisaron tipos, ejemplos,
histogramas, KDE, correlaciones con elipses, PCA estandarizado con imputación por
mediana y pares correlacionados mediante densidades y violines. Es un mapa inicial,
no evidencia de causalidad ni de capacidad predictiva. Los violines describen
distribuciones marginales y no la relación emparejada entre dos variables.

- Hay cuatro grupos de origen: CFI-G normal (179), CFI-G inversa (154), Flame
  Design normal (207) y Flame Design inversa (120). Campaña y combustible están
  parcialmente confundidos: etileno solo aparece en Flame Design; etano y butano,
  solo en CFI-G. Por ello una correlación global puede reflejar mezcla de diseños.
- La etiqueta `ignited` no tiene una fuente equivalente entre campañas: CFI-G
  conserva `Hot ignition`; Flame Design deriva `Ignite` de EventData. Flame Design
  tiene 323 positivos y dos negativos entre 325 etiquetas conocidas; se conserva
  como contexto, sin combinarlo con CFI-G para el clasificador inicial.
- Los ocho `flash` pertenecen a CFI-G inversa; se preservan como categoría y no se
  recodifican a positivo/negativo. No equiparar ignición con llama sostenida.
- Normal e inversa intercambian el papel de la corriente del quemador y el ambiente.
  `xo2`, `x_fuel` y los caudales deben interpretarse según geometría. En particular,
  `po2_kpa` no se interpreta automáticamente como oxígeno de toda la cámara en inversas.
- Los faltantes son parcialmente estructurales: `m_n2_mg_s` solo se incorpora desde
  Flame Design y los caudales de combustible/oxígeno dependen de la geometría.
  `RAMP`, `RAMPED` y `LOW O2` describen condiciones especiales, no valores numéricos
  para rellenar con una mediana. `cool_flame` tiene solo ocho respuestas informadas.
- `po2_kpa` se calcula como `xo2 * p_kpa`; `zst` y `tad_k` son cantidades calculadas.
  Identificadores, `tier`, etiquetas y resultados posteriores a la prueba no son
  entradas intercambiables con condiciones iniciales. El PCA global anterior puede
  reflejar el patrón de faltantes y la imputación; no se usa para seleccionar predictores.
- En la selección previa, `categorical_data` se sobrescribe con booleanos. Las nuevas
  celdas crean objetos independientes y distinguen booleanos anulables explícitamente.

### Reglas operativas, fijadas antes del análisis segmentado

1. **Auditar el merge y la identidad.** Recargar gold y unir metadatos de investigación
   con `validate='many_to_one'`. Normalizar espacios/mayúsculas en los identificadores.
   Una copia solo se retira si comparte identidad y todos los campos científicos;
   cualquier diferencia científica bajo la misma identidad se deja en cuarentena.
   Se conserva el registro de filas retiradas y su procedencia.
2. **Verificar cronología desde silver/EventData.** Usar `TP` para normales e `INV_TP`
   para inversas y unir por investigación/prueba sin multiplicar filas. Si EventData
   tiene versiones incompatibles de una prueba, no elegir arbitrariamente la primera.
   Conservar archivo/fila de ambos orígenes, inicio/fin GMT y banderas de calidad.
   Fechas desconocidas siguen desconocidas; no se deducen del identificador.
3. **Distinguir duplicados, jornadas y réplicas.** Mismo día no significa misma prueba.
   Dos IDs distintos con condiciones registradas idénticas son candidatos a réplica,
   no duplicados demostrados. Se conservan para estimar variabilidad. La firma compara
   valores exactos, sin redondeo, con cobertura completa y dentro de una familia;
   no garantiza equivalencia de condiciones no registradas. Los casos cercanos pero
   no idénticos requieren tolerancias físicas acordadas antes de agruparlos.
4. **Segmentación jerárquica.** Familia base = campaña × tipo de llama × combustible.
   Familia estricta añade porcentaje de combustible de botella, año observado en
   EventData, protocolo y origen de etiqueta. El porcentaje de botella no sustituye
   `x_fuel`. `no_informada` y `sin_fecha` se mantienen explícitos, sin asumir 100 %.
   Año es un bloqueo inicial de procedencia, no prueba de equivalencia instrumental.
   Flujo cero se etiqueta aparte como posible control; no se borra ni se supone que
   todos los registros sin qualifier sean estacionarios.
5. **Control de calidad conservador.** Conflictos de identidad, operación incompatible,
   colisión temporal exacta o final anterior al inicio se apartan del análisis principal
   y permanecen disponibles para revisión/sensibilidad. Una fecha ausente, por sí sola,
   no invalida una prueba ni prueba duplicación. No se corrige la fuente desde el notebook.
6. **EDA condicional.** Describir todas las familias y faltantes; separar protocolos
   especiales y controles. Recalcular distribuciones y correlaciones dentro de familias.
   Pearson/Spearman requieren al menos 20 pares y variación. El nuevo PCA usa condiciones
   aplicables por geometría, estandarización y casos completos, sin imputación estructural.
   Los grupos pequeños o constantes se documentan, no se fuerzan a un PCA.
7. **Dependencia y validación futura.** No declarar independencia por haber eliminado
   duplicados. Mantener bloques de operación y secuencias relacionadas. El prefijo del ID
   se usa como token nominal del bloque, nunca como fecha UTC. Las jornadas pueden cruzar
   medianoche; `test_day` se conserva y se audita. La validación y los intervalos futuros
   deben agrupar por operación/campaña, con sensibilidad por fuente/periodo. Aquí se hace
   análisis descriptivo: no se presentan pruebas de hipótesis ni p-valores exploratorios.

Las fuentes, conteos y anomalías se calculan de nuevo en las celdas siguientes.
Los archivos gold/silver se leen sin modificarlos. Los resultados auxiliares se guardan
en `artifacts/eda_segmentation/` y no sustituyen los datos originales.

In [ ]:
from pathlib import Path
from itertools import combinations
import hashlib
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from flamesafe import config
from flamesafe.analysis.segmentation import (
    audit_gas_observations, family_summary, condition_candidates,
)

sns.set_theme(style="whitegrid", context="notebook")
ROOT_EDA = config.ROOT
DATA_EDA = config.data_dir()
OUT_EDA = ROOT_EDA / "artifacts" / "eda_segmentation"
OUT_EDA.mkdir(parents=True, exist_ok=True)
SEED_EDA = 42
MIN_PARES_EDA = 20

archivos_eda = [
    DATA_EDA / "gold" / "gas_flame_tests.parquet",
    DATA_EDA / "gold" / "investigations.parquet",
    DATA_EDA / "silver" / "tables" / "psi159__eventdata.parquet",
    DATA_EDA / "silver" / "tables" / "psi10__eventdata.parquet",
]
for archivo in archivos_eda:
    if not archivo.exists():
        raise FileNotFoundError(f"Falta {archivo}; reconstruir el pipeline antes del EDA.")

gas_origen_eda = pd.read_parquet(archivos_eda[0])
investigaciones_eda = pd.read_parquet(archivos_eda[1])
metadatos_eda = ["psi_id", "acronym", "tier", "hardware_family", "environments"]
gas_base_segmentacion = gas_origen_eda.merge(
    investigaciones_eda[metadatos_eda], on="psi_id", how="left",
    validate="many_to_one", indicator="_metadata_merge",
)
assert gas_base_segmentacion["_metadata_merge"].eq("both").all(), "Metadatos sin correspondencia"
gas_base_segmentacion = gas_base_segmentacion.drop(columns="_metadata_merge")
eventos_eda = pd.concat([pd.read_parquet(p) for p in archivos_eda[2:]], ignore_index=True)
auditoria_gas = audit_gas_observations(gas_base_segmentacion, eventos_eda)
gas_auditado = auditoria_gas.observations
gas_unicos = auditoria_gas.unique
gas_segmentado = gas_unicos.loc[gas_unicos["audit_ok"]].copy()

mapa_familias = pd.DataFrame({"family_id": sorted(gas_auditado["family_id"].unique())})
mapa_familias["family_code"] = [f"F{i:02d}" for i in range(1, len(mapa_familias) + 1)]
codigo_familia = mapa_familias.set_index("family_id")["family_code"]
for tabla in [gas_auditado, gas_unicos, gas_segmentado]:
    tabla["family_code"] = tabla["family_id"].map(codigo_familia)

manifiesto_eda = pd.DataFrame([{
    "file": str(p.relative_to(DATA_EDA)), "bytes": p.stat().st_size,
    "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
} for p in archivos_eda])
display(manifiesto_eda)
display(auditoria_gas.summary)
assert len(gas_auditado) == len(gas_base_segmentacion), "El enlace temporal multiplicó filas"
assert not gas_unicos.duplicated(["psi_id", "test_id"]).any()

### Identidad, fechas y casos que requieren revisión

La tabla siguiente separa ausencia de timestamp, inconsistencias y copias confirmadas.
`duration_event_s` es la duración del evento operativo, **no** `burn_time_s`.
No se restaura un fin incorrecto sumándole un día sin contrastar la fuente.

In [ ]:
columnas_revision = [
    "psi_id", "test_id", "campaign", "flame_type", "test_day", "identity_status",
    "t_start_gmt", "t_end_gmt", "duration_event_s", "event_op_mismatch",
    "negative_event_duration", "event_identity_conflict", "temporal_collision",
    "source_file", "source_row", "event_source_file", "event_source_row",
]
gas_revision = gas_auditado.loc[~gas_auditado["audit_ok"], columnas_revision]
gas_sin_timestamp = gas_unicos.loc[gas_unicos["t_start_gmt"].isna(), columnas_revision]
display(Markdown("**Anomalías o conflictos (conservados para revisión):**"))
display(gas_revision)
display(Markdown("**Pruebas sin timestamp verificable:**"))
display(gas_sin_timestamp)
display(Markdown("**Copias idénticas retiradas, si las hay:**"))
display(auditoria_gas.removed_copies)
if not auditoria_gas.event_conflicts.empty:
    display(auditoria_gas.event_conflicts)

# Ninguna de estas filas se elimina solo por compartir fecha.
por_fecha_utc = (
    gas_unicos.dropna(subset=["utc_date"])
    .groupby(["psi_id", "campaign", "flame_type", "utc_date"], dropna=False)
    .agg(n_pruebas=("test_id", "nunique"), bloques=("operation_group", "nunique"))
    .reset_index()
)
por_operacion = (
    gas_unicos.groupby("operation_group")
    .agg(n_pruebas=("test_id", "nunique"), n_test_day=("test_day", "nunique"),
         n_fechas_utc=("utc_date", "nunique"),
         inicio=("t_start_gmt", "min"), fin=("t_start_gmt", "max"))
    .reset_index()
)
display(Markdown("**Bloques con pruebas en más de una fecha UTC:**"))
display(por_operacion.loc[por_operacion["n_fechas_utc"] > 1])
if por_operacion["n_test_day"].gt(1).any():
    print("ATENCIÓN: revisar bloques que reúnen varios test_day antes de validar modelos.")

fig, ax = plt.subplots(figsize=(13, 4))
for (campana, tipo), g in por_fecha_utc.groupby(["campaign", "flame_type"]):
    ax.scatter(pd.to_datetime(g["utc_date"]), [f"{campana} | {tipo}"] * len(g),
               s=15 + 7 * g["n_pruebas"], alpha=0.6)
ax.set(title="Cobertura temporal: tamaño del punto proporcional al número de pruebas",
       xlabel="Fecha UTC de inicio", ylabel="Grupo de origen")
plt.tight_layout()
plt.show()

### Familias y cohortes explícitas

Se mantienen todas las pruebas únicas y se construyen vistas analíticas separadas.
La cohorte binaria inicial excluye `flash`, posibles controles de flujo cero y
anomalías de auditoría; esas observaciones siguen disponibles en tablas de revisión.
Las fechas ausentes se mantienen en familias `sin_fecha` y se evalúan por sensibilidad.
La dilución de botella es una etiqueta de procedencia, no una fracción medida en cámara.

In [ ]:
segmentos_gas = {nombre: g.copy() for nombre, g in gas_segmentado.groupby("family_id")}
gas_cfig_eda = gas_segmentado.loc[
    gas_segmentado["campaign"].eq("CFI-G")
    & gas_segmentado["protocol_group"].eq("sin_anotacion")
].copy()
gas_cfig_binario = gas_segmentado.loc[gas_segmentado["is_cfig_binary"]].copy()
gas_flash = gas_segmentado.loc[gas_segmentado["ignition_outcome"].eq("flash")].copy()
gas_contexto_fd = gas_segmentado.loc[gas_segmentado["campaign"].eq("Flame Design")].copy()
gas_protocolos = gas_segmentado.loc[~gas_segmentado["protocol_group"].eq("sin_anotacion")].copy()

resumen_familias = family_summary(gas_segmentado).merge(mapa_familias, on="family_id", validate="one_to_one")
display(resumen_familias[["family_code", "family_id", "n", "operation_groups", "without_timestamp",
                          "yes", "no", "flash", "missing_label", "yes_pct_binary"]])
display(pd.DataFrame({
    "vista": ["Original", "Únicos", "Sin anomalías", "CFI-G EDA sin controles",
              "CFI-G binario", "Flash", "Contexto Flame Design", "Protocolos/controles"],
    "n": [len(gas_base_segmentacion), len(gas_unicos), len(gas_segmentado), len(gas_cfig_eda),
          len(gas_cfig_binario), len(gas_flash), len(gas_contexto_fd), len(gas_protocolos)],
}))
print("Las vistas se superponen: no sumar sus tamaños como si fueran particiones.")

resultados_origen = pd.crosstab(
    [gas_unicos["campaign"], gas_unicos["flame_type"]],
    gas_unicos["ignition_outcome"].fillna("sin_resultado"),
).reindex(columns=["yes", "no", "flash", "sin_resultado"], fill_value=0)
display(resultados_origen)
resultados_origen.plot.bar(stacked=True, figsize=(11, 4),
                           color=["#2878B5", "#D55E00", "#E8B339", "#999999"])
plt.ylabel("Número de pruebas únicas")
plt.xlabel("Campaña y tipo de llama")
plt.title("Balance observado: las etiquetas de las campañas no se presuponen equivalentes")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

### Condiciones repetidas: candidatos a réplica y dependencia

Se buscan coincidencias exactas en condiciones registradas completas dentro de
la familia estricta. Cambiar de ID o de hora distingue ensayos; no se colapsan sus
resultados. Las réplicas pueden revelar variabilidad real o condiciones omitidas.
`hc_burnrate_mg_s` se usa solo para contrastar registros inversos CFI-G en esta
auditoría; su semántica pendiente impide tratarlo como caudal de alimentación en PCA.

In [ ]:
candidatos_replicas = condition_candidates(gas_segmentado)
if candidatos_replicas.empty:
    grupos_replicas = pd.DataFrame()
    print("No hay coincidencias exactas completas en las condiciones seleccionadas.")
else:
    grupos_replicas = (
        candidatos_replicas.groupby(["family_id", "settings"])
        .agg(n_pruebas=("test_id", "size"),
             n_bloques=("operation_group", "nunique"),
             n_resultados=("ignition_outcome", "nunique"),
             pruebas=("test_id", lambda s: ", ".join(s)))
        .reset_index()
    )
    grupos_replicas["family_code"] = grupos_replicas["family_id"].map(codigo_familia)
    display(grupos_replicas[["family_code", "n_pruebas", "n_bloques", "n_resultados", "pruebas", "settings"]])
    print(f"{len(candidatos_replicas)} filas en {len(grupos_replicas)} grupos de condiciones idénticas.")
    print("No se retira ninguna: son candidatos a réplica, no copias del mismo experimento.")
display(gas_flash[["psi_id", "test_id", "family_code", "fuel", "xo2", "p_kpa",
                   "burn_time_s", "extinction_mode", "t_start_gmt"]])

### Faltantes y variables aplicables

Las columnas de la matriz siguiente se seleccionan explícitamente como condiciones
o resultados. Identificadores y metadatos numéricos no se grafican como mediciones.
El nuevo EDA no rellena faltantes estructurales. Una columna nula no significa flujo cero.

In [ ]:
variables_cobertura = [
    "xo2", "p_kpa", "x_fuel", "m_fuel_mg_s", "m_o2_mg_s", "m_n2_mg_s",
    "zst", "tad_k", "ignited", "burn_time_s", "cool_flame",
]
faltantes_por_origen = (
    gas_unicos.groupby(["campaign", "flame_type"])[variables_cobertura]
    .agg(lambda s: 100 * s.isna().mean())
).astype(float)
fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(faltantes_por_origen, annot=True, fmt=".0f", vmin=0, vmax=100,
            cmap="YlOrRd", cbar_kws={"label": "% faltantes"}, ax=ax)
ax.set_title("Ausencia de datos por grupo experimental")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

variables_por_geometria = {
    "normal": ["xo2", "p_kpa", "x_fuel", "m_fuel_mg_s"],
    "inverse": ["xo2", "p_kpa", "x_fuel", "m_o2_mg_s"],
}
mediciones_eda = gas_segmentado.select_dtypes(include="number").drop(
    columns=["test_day", "source_row", "tier", "_input_row", "event_source_row"], errors="ignore"
)
booleanos_eda = gas_segmentado.select_dtypes(include=["bool", "boolean"])
perfil_familias = []
for familia, g in segmentos_gas.items():
    for variable in variables_por_geometria[g["flame_type"].iloc[0]]:
        valores = g[variable].replace([np.inf, -np.inf], np.nan)
        perfil_familias.append({
            "family_id": familia, "family_code": g["family_code"].iloc[0],
            "variable": variable, "n": len(g), "validos": valores.notna().sum(),
            "faltantes_pct": valores.isna().mean() * 100, "n_unicos": valores.nunique(),
            "min": valores.min(), "mediana": valores.median(), "max": valores.max(),
        })
perfil_familias = pd.DataFrame(perfil_familias)
display(perfil_familias.round(3))

### Distribuciones condicionales de CFI-G

Las curvas acumuladas empíricas (ECDF) permiten comparar cobertura sin suavizar
fuera de límites físicos. Cada curva mantiene combustible, dilución y periodo.
Las muestras de una fila se muestran como puntos y no como densidades inferidas.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9), layout="constrained")
for fila, geometria in enumerate(["normal", "inverse"]):
    g_tipo = gas_cfig_eda.loc[gas_cfig_eda["flame_type"].eq(geometria)]
    for ax, variable in zip(axes[fila], ["xo2", "p_kpa", "x_fuel"]):
        for familia, g in g_tipo.groupby("family_id"):
            valores = np.sort(g[variable].dropna().to_numpy())
            if not len(valores):
                continue
            etiqueta = f"{g['family_code'].iloc[0]}: {g['fuel'].iloc[0]}, {g['dilution_group'].iloc[0]}, {g['period_group'].iloc[0]} (n={len(valores)})"
            ax.step(valores, np.arange(1, len(valores) + 1) / len(valores), where="post", label=etiqueta)
            if len(valores) == 1:
                ax.plot(valores, [1], "o")
        ax.set(title=f"CFI-G {geometria}: {variable}", xlabel=variable, ylabel="Fracción acumulada", ylim=(0, 1.05))
        ax.legend(fontsize=7, loc="best")
plt.show()

# Histogramas/KDE por familia: distribución del diseño experimental, no del riesgo poblacional.
familias_graficables = [
    (nombre, g) for nombre, g in gas_cfig_eda.groupby("family_id") if len(g) >= MIN_PARES_EDA
]
for comienzo in range(0, len(familias_graficables), 3):
    lote = familias_graficables[comienzo:comienzo + 3]
    fig, axes = plt.subplots(len(lote), 3, figsize=(15, 3.2 * len(lote)), squeeze=False, layout="constrained")
    for fila, (familia, g) in enumerate(lote):
        geometria = g["flame_type"].iloc[0]
        flujo = variables_por_geometria[geometria][-1]
        for ax, variable in zip(axes[fila], ["xo2", "p_kpa", flujo]):
            valores = g[variable].replace([np.inf, -np.inf], np.nan).dropna()
            sns.histplot(x=valores, stat="density", bins=min(15, max(4, valores.nunique())), ax=ax, color="#2878B5")
            if valores.nunique() >= 5 and len(valores) >= MIN_PARES_EDA:
                sns.kdeplot(x=valores, cut=0, color="#D55E00", ax=ax, warn_singular=False)
            ax.set(title=f"{g['family_code'].iloc[0]} | {geometria} | {g['fuel'].iloc[0]} | {g['dilution_group'].iloc[0]}\n{variable}: n={len(valores)}", ylabel="Densidad", xlabel=variable)
    plt.show()

### Correlaciones dentro de familias y contraste con la mezcla global

Solo condiciones aplicables por geometría, sin resultados ni `po2_kpa` derivada.
Cada coeficiente informa su número de pares completos; Pearson y Spearman se
calculan sin imputación. Las correlaciones siguen siendo descriptivas y pueden
reflejar condiciones elegidas conjuntamente. No se comparan tamaños de coeficientes
como si provinieran de muestras independientes ni se afirma una paradoja de Simpson.

In [ ]:
filas_corr = []
for familia, g in gas_segmentado.loc[gas_segmentado["protocol_group"].eq("sin_anotacion")].groupby("family_id"):
    variables = variables_por_geometria[g["flame_type"].iloc[0]]
    for a, b in combinations(variables, 2):
        par = g[[a, b]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(par) < MIN_PARES_EDA or par.nunique().min() < 2:
            continue
        global_par = gas_segmentado[[a, b]].replace([np.inf, -np.inf], np.nan).dropna()
        filas_corr.append({
            "family_id": familia, "family_code": g["family_code"].iloc[0],
            "variable_1": a, "variable_2": b, "n": len(par),
            "n_operaciones": g.loc[par.index, "operation_group"].nunique(),
            "pearson": par[a].corr(par[b]), "spearman": par[a].corr(par[b], method="spearman"),
            "pearson_mezcla_global": global_par[a].corr(global_par[b]),
            "n_mezcla_global": len(global_par),
        })
correlaciones_familia = pd.DataFrame(filas_corr)
if not correlaciones_familia.empty:
    correlaciones_familia["abs_r"] = correlaciones_familia["pearson"].abs()
    correlaciones_familia = correlaciones_familia.sort_values("abs_r", ascending=False).reset_index(drop=True)
    display(correlaciones_familia.drop(columns="family_id").round(3))

    seleccion_corr = correlaciones_familia.loc[correlaciones_familia["abs_r"] > 0.60].head(6)
    if len(seleccion_corr):
        fig, axes = plt.subplots(math.ceil(len(seleccion_corr) / 3), 3,
                                 figsize=(16, 4 * math.ceil(len(seleccion_corr) / 3)),
                                 squeeze=False, layout="constrained")
        for ax, r in zip(axes.flat, seleccion_corr.itertuples(index=False)):
            g = segmentos_gas[r.family_id]
            sns.scatterplot(data=g, x=r.variable_1, y=r.variable_2,
                            hue="ignition_outcome", hue_order=["yes", "no", "flash"],
                            palette={"yes": "#2878B5", "no": "#D55E00", "flash": "#E8B339"}, ax=ax, alpha=0.75)
            ax.set_title(f"{r.family_code}: r={r.pearson:.2f}, rho={r.spearman:.2f}\nn={r.n}; operaciones={r.n_operaciones}")
        for ax in axes.flat[len(seleccion_corr):]:
            ax.set_visible(False)
        plt.show()
else:
    print("No hay familias con suficientes pares variables para correlación.")

### PCA por familia: sin imputación y con cobertura explícita

Se exige al menos 80 % de cobertura por variable, variación en la familia y al menos
20 casos completos. Se vuelve a comprobar variación y rango matricial después del
filtro. No se rellenan caudales no aplicables. Familias con una sola operación no
permiten evaluar generalización entre operaciones, aunque produzcan un PCA.
El PCA representa variación del diseño y no importancia causal para la ignición.

In [ ]:
resultados_pca_familia = {}
resumen_pca_filas = []
diagnostico_pca = []
for familia, g in gas_cfig_eda.groupby("family_id"):
    candidatas = variables_por_geometria[g["flame_type"].iloc[0]]
    x = g[candidatas].replace([np.inf, -np.inf], np.nan).astype(float)
    columnas = [c for c in x if x[c].notna().mean() >= 0.80 and x[c].nunique() > 1]
    completos = x[columnas].dropna()
    completos = completos.loc[:, completos.nunique() > 1]
    razon = "apta"
    if completos.shape[1] < 2:
        razon = "menos de dos condiciones con variación"
    elif len(completos) < MIN_PARES_EDA:
        razon = "menos de 20 casos completos"
    else:
        z = StandardScaler().fit_transform(completos)
        if np.linalg.matrix_rank(z) < 2:
            razon = "rango matricial menor de dos"
    diagnostico_pca.append({"family_code": g["family_code"].iloc[0], "n_familia": len(g),
                           "n_completos": len(completos), "variables": ", ".join(completos.columns), "estado": razon})
    if razon != "apta":
        continue
    modelo = PCA().fit(z)
    acumulada = modelo.explained_variance_ratio_.cumsum()
    pesos = pd.DataFrame(modelo.components_.T, index=completos.columns,
                         columns=[f"PC{i + 1}" for i in range(len(acumulada))])
    resultados_pca_familia[familia] = {"modelo": modelo, "pesos": pesos, "indices": completos.index,
                                      "scores": modelo.transform(z), "acumulada": acumulada}
    resumen_pca_filas.append({
        "family_id": familia, "family_code": g["family_code"].iloc[0],
        "n_familia": len(g), "n_completos": len(completos),
        "operaciones": g.loc[completos.index, "operation_group"].nunique(),
        "variables": ", ".join(completos.columns),
        "pc1_pct": 100 * modelo.explained_variance_ratio_[0],
        "k90": min(len(acumulada), int(np.searchsorted(acumulada, 0.90)) + 1),
    })
resumen_pca = pd.DataFrame(resumen_pca_filas)
diagnostico_pca = pd.DataFrame(diagnostico_pca)
display(diagnostico_pca)
display(resumen_pca.round(2))
if resultados_pca_familia:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")
    for familia, resultado in resultados_pca_familia.items():
        pc = np.arange(1, len(resultado["acumulada"]) + 1)
        axes[0].plot(pc, 100 * resultado["modelo"].explained_variance_ratio_, "o-", label=codigo_familia[familia])
        axes[1].plot(pc, 100 * resultado["acumulada"], "o-", label=codigo_familia[familia])
    axes[0].set(title="Varianza por componente dentro de cada familia", xlabel="Componente", ylabel="Varianza (%)")
    axes[1].set(title="Varianza acumulada por familia", xlabel="Componentes", ylabel="Varianza (%)", ylim=(0, 105))
    axes[1].axhline(90, ls="--", color="gray")
    for ax in axes:
        ax.legend(fontsize=8)
    plt.show()
    for familia, resultado in resultados_pca_familia.items():
        print(codigo_familia[familia], "- pesos de los dos primeros componentes")
        display(resultado["pesos"].iloc[:, :2].round(3))

### Sensibilidad, conclusiones y trazabilidad

No se descartan posibles réplicas para obtener artificialmente independencia.
Las proporciones de positivos siguientes son descriptivas: el promedio por operación
asigna igual peso a cada bloque, mientras el promedio por prueba pesa más los bloques
con más ensayos. No son estimaciones de riesgo para una población de misiones.
Se compara también la inclusión del registro con cronología incorrecta, sin corregirlo,
y la exclusión de fechas no verificadas. `flash` continúa fuera del objetivo binario.

In [ ]:
base_sensibilidad = gas_unicos.loc[
    gas_unicos["campaign"].eq("CFI-G")
    & gas_unicos["outcome_source"].eq("experimental_table")
    & gas_unicos["ignited"].notna()
    & gas_unicos["protocol_group"].eq("sin_anotacion")
    & ~gas_unicos["event_identity_conflict"] & ~gas_unicos["event_op_mismatch"]
    & ~gas_unicos["temporal_collision"]
]
escenarios = {
    "Principal: CFI-G binario sin controles ni anomalías": gas_cfig_binario,
    "Solo timestamps verificados": gas_cfig_binario.loc[gas_cfig_binario["timestamp_verified"]],
    "Sensibilidad: incluir fin anterior al inicio": base_sensibilidad,
    "Sensibilidad: incluir controles de flujo cero": gas_segmentado.loc[
        gas_segmentado["campaign"].eq("CFI-G") & gas_segmentado["ignited"].notna()
        & gas_segmentado["protocol_group"].isin(["sin_anotacion", "control_flujo_cero"])
    ],
}
filas_sensibilidad = []
for nombre, g in escenarios.items():
    for geometria, sub in g.groupby("flame_type"):
        por_bloque = sub.groupby("operation_group")["ignited"].mean()
        filas_sensibilidad.append({
            "escenario": nombre, "flame_type": geometria, "n": len(sub),
            "operaciones": len(por_bloque), "positivos_pct_por_prueba": 100 * sub["ignited"].mean(),
            "positivos_pct_igual_peso_operacion": 100 * por_bloque.mean(),
        })
sensibilidad = pd.DataFrame(filas_sensibilidad)
display(sensibilidad.round(2))

tablas_exportar = {
    "observaciones_auditadas": gas_auditado, "observaciones_unicas": gas_unicos,
    "familias": resumen_familias, "revision": gas_revision,
    "copias_retiradas": auditoria_gas.removed_copies,
    "conflictos_identidad": auditoria_gas.identity_conflicts,
    "conflictos_eventdata": auditoria_gas.event_conflicts,
    "sin_timestamp": gas_sin_timestamp, "bloques_operacion": por_operacion,
    "candidatos_replicas": candidatos_replicas, "grupos_condiciones": grupos_replicas,
    "perfil_familias": perfil_familias, "correlaciones_familia": correlaciones_familia,
    "pca_familias": resumen_pca, "pca_diagnostico": diagnostico_pca,
    "sensibilidad": sensibilidad, "fuentes": manifiesto_eda,
}
for nombre, tabla in tablas_exportar.items():
    tabla.to_csv(OUT_EDA / f"{nombre}.csv", index=False, encoding="utf-8-sig")
reporte_eda = {
    "inputs": manifiesto_eda.to_dict(orient="records"),
    "audit": auditoria_gas.summary["n"].astype(int).to_dict(),
    "replica_candidate_rows": len(candidatos_replicas),
    "matching_settings_groups": len(grupos_replicas),
    "binary_cfig_rows": len(gas_cfig_binario),
    "flash_rows": len(gas_flash),
    "families_with_pca": len(resultados_pca_familia),
    "rules": {"min_pairs": MIN_PARES_EDA, "pca_min_coverage": 0.8,
              "dates_imputed": False, "structural_missing_imputed": False,
              "replicas_removed": False, "grouping": "operation_group"},
}
(OUT_EDA / "resumen.json").write_text(json.dumps(reporte_eda, indent=2, ensure_ascii=False), encoding="utf-8")

display(Markdown(f"""
### Resultado de esta ejecución

- **{len(gas_base_segmentacion)} filas originales; {len(gas_unicos)} identidades únicas.**
  Copias idénticas retiradas: **{len(auditoria_gas.removed_copies)}**.
- **{len(gas_revision)} filas requieren revisión**; **{len(gas_sin_timestamp)}** no tienen
  timestamp de inicio verificable. No se inventan fechas ni se corrige EventData.
- Se conservan **{len(candidatos_replicas)} pruebas** con condiciones coincidentes en
  **{len(grupos_replicas)} grupos candidatos a réplica**. Mismo día/condición no prueba duplicación.
  En **{int(grupos_replicas['n_resultados'].gt(1).sum()) if not grupos_replicas.empty else 0} grupos**
  hay resultados distintos con las mismas condiciones registradas: eliminarlos borraría
  información sobre variabilidad, condiciones omitidas o diferencias del protocolo.
- La partición estricta deja **{len(resumen_familias)} familias sin anomalías** y
  **{gas_segmentado['operation_group'].nunique()} bloques de operación**. Familia no equivale a independencia.
  **{int(resumen_familias['n'].lt(MIN_PARES_EDA).sum())} familias** tienen menos de
  {MIN_PARES_EDA} pruebas y **{int(resumen_familias['operation_groups'].eq(1).sum())}**
  provienen de una sola operación. No deben tratarse como conjuntos suficientes para
  validación independiente, aunque su correlación sea fuerte.
- La vista binaria inicial CFI-G contiene **{len(gas_cfig_binario)} pruebas** y mantiene
  fuera **{len(gas_flash)} flash**. Flame Design se conserva como contexto.
- PCA pudo ejecutarse en **{len(resultados_pca_familia)} familias** con cobertura y variación
  suficientes. Sus componentes describen el diseño; no seleccionan causas de ignición.

**Pendiente de revisión física:** significado de `flash`, equivalencia de `Ignite`,
semántica de `HC burnrate`, interpretación de `po2_kpa` en inversas y tolerancias para
considerar condiciones cercanas como réplicas. Antes de entrenar: validar agrupando
por operación y revisar combustible/dilución/periodo como posibles confusores.

Tablas auditables: `{OUT_EDA}`. Los conjuntos `segmentos_gas`, `gas_cfig_eda`,
`gas_cfig_binario`, `gas_flash` y `gas_contexto_fd` quedan disponibles en memoria.
"""))

<!-- FLAMESAFE:CATEGORICAL:BEGIN -->
## Variables categóricas frente a numéricas: asociaciones condicionales

**Protocolo descriptivo fijado antes de esta ejecución · 2026-11-15.** Esta sección
conserva el EDA anterior y reutiliza su auditoría. El objetivo es describir diferencias,
cobertura y posibles confusores; no calcular significancia, intervalos, riesgos de misión
ni modelos predictivos. «Interacción» significa aquí asociación condicional explorada.

### Preguntas y roles de las variables

| Papel | Variables | Pregunta y uso |
|---|---|---|
| Diseño | `fuel`, `dilution_group` | ¿Qué condiciones se exploraron bajo cada categoría, fijando los demás factores? |
| Geometría | `flame_type` | Paneles separados: cambia la corriente del quemador y la interpretación de composición/caudal. |
| Procedencia/protocolo | `campaign`, `period_group`, `protocol_group`, `outcome_source` | Cobertura, composición y faltantes; no efectos físicos de campaña. |
| Ignición | `ignition_outcome`, `ignited` | ¿Cómo se distribuyen las condiciones entre `yes`, `no`, `flash` dentro de CFI-G? |
| Resultados posteriores | `extinction_mode`, `burn_time_s`, `peak_burner_t_k` | Descripción secundaria, solo CFI-G con ignición positiva. |
| Escasos/no definidos | `cool_flame`, `tfp` | Frecuencias, faltantes y registros; no comparar efectos de un flag no definido. |
| Dependencia/identidad | `test_id`, `operation_group`, fecha | Conteo de operaciones y trazabilidad, no mediciones físicas. |

Condiciones principales: `xo2`, `p_kpa`, `x_fuel` y `m_fuel_mg_s` en normales o
`m_o2_mg_s` en inversas. `m_n2_mg_s` solo se explora en Flame Design. `po2_kpa`,
`zst` y `tad_k` se presentan aparte como cantidades derivadas/calculadas; `po2_kpa`
en inversas no se supone oxígeno de toda la cámara. El porcentaje de combustible
de botella no se interpreta automáticamente como `x_fuel` de la mezcla experimental.

### Comparabilidad y cohortes

- **Combustible:** bloquear campaña, geometría, dilución, periodo, protocolo y fuente
  de etiqueta. **Dilución:** fijar además combustible y retirar dilución del bloque.
  Comparar una categoría exige que pueda variar; no se usa `family_id` fijo para ello.
- **Ignición:** comparar categorías dentro de la familia estricta CFI-G. No recodificar
  los ocho `flash`. Flame Design es contexto; sus etiquetas no se homologan con CFI-G.
- **Extinción:** solo CFI-G, ignición positiva y modo informado. Un final por corte de
  flujo termina la observación; no representa el tiempo de extinción natural. Duraciones
  cortas o largas entre SE/FT no se interpretan como eficiencia causal de supresión.
- Principal: filas únicas sin anomalías y sin protocolos especiales ni flujo cero.
  Las cuatro fechas ausentes no se rellenan; se repite el resumen sin ellas.
  Controles, rampas, `LOW O2` y la cronología anómala se documentan por separado.
- No eliminar posibles réplicas, redondear condiciones, muestrear filas ni imputar datos.
  Los valores no finitos cuentan como ausentes; los ceros se conservan.

### Reglas de resumen y visualización

Cada celda categoría × bloque × variable informa n total/válido/faltante, operaciones,
mediana, cuartiles y rango. Se muestran puntos individuales; boxplots desde cinco valores
y violines solo desde 20 valores y cinco valores distintos. Las curvas ECDF no suavizan
fuera del soporte observado. Los rombos representan medianas por operación.

Para pares con al menos cinco valores válidos por categoría se informa diferencia de
medianas **A − B** y delta de Cliff **P(A>B) − P(A<B)**. Son efectos descriptivos de estas
pruebas, sin umbrales de significancia ni de importancia causal. Las categorías pequeñas,
constantes o ausentes se identifican; su falta de soporte no se esconde con una prueba.

El solapamiento de rangos se evalúa para cada variable por separado. No demuestra
comparabilidad multidimensional, control de confusión ni intercambiabilidad de categorías.
La mediana por observación y la mediana de medianas por operación tienen ponderaciones
diferentes. Una familia de una operación no permite concluir generalización entre jornadas.
Se informa además cuántas operaciones comparten las categorías de cada contraste:
varias operaciones por categoría no eliminan la confusión si pertenecen a jornadas disjuntas.

Todas las tablas y las figuras pertinentes se guardan en `artifacts/eda_categoricas_numericas/`.
El notebook muestra un máximo configurable de cuatro figuras por pregunta, por cobertura
y diversidad de bloques, no por magnitud del efecto. El manifiesto conserva las demás.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from flamesafe import config
from flamesafe.analysis.categorical import (
    describe_categories, plot_comparison, PRIMARY_VARIABLES, ORDERS,
)

requeridos_cn = ["gas_auditado", "gas_unicos", "gas_segmentado"]
if any(nombre not in globals() for nombre in requeridos_cn):
    raise RuntimeError("Ejecuta primero la sección de segmentación, o Restart & Run All.")
assert not gas_unicos.duplicated(["psi_id", "test_id"]).any()
OUT_CN = config.ROOT / "artifacts" / "eda_categoricas_numericas"
FIG_CN = OUT_CN / "figures"
FIG_CN.mkdir(parents=True, exist_ok=True)
MAX_FIGURAS_EN_NOTEBOOK_CN = 4
sns.set_theme(style="whitegrid", context="notebook")

entradas_cn = [config.data_dir() / p for p in [
    "gold/gas_flame_tests.parquet", "gold/investigations.parquet",
    "silver/tables/psi159__eventdata.parquet", "silver/tables/psi10__eventdata.parquet",
]]
fuentes_cn = pd.DataFrame([{
    "archivo": str(p.relative_to(config.data_dir())), "bytes": p.stat().st_size,
    "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
} for p in entradas_cn])
display(fuentes_cn)
principal_cn = gas_segmentado.loc[gas_segmentado["protocol_group"].eq("sin_anotacion")].copy()
cfig_cn = principal_cn.loc[principal_cn["campaign"].eq("CFI-G")].copy()
fd_cn = principal_cn.loc[principal_cn["campaign"].eq("Flame Design")].copy()
extincion_cn = cfig_cn.loc[
    cfig_cn["ignited"].fillna(False) & cfig_cn["extinction_mode"].notna()
].copy()
cohortes_cn = pd.DataFrame([
    {"cohorte": nombre, "n_pruebas": len(g), "n_operaciones": g["operation_group"].nunique(),
     "n_familias": g["family_id"].nunique(), "sin_timestamp": int(g["t_start_gmt"].isna().sum())}
    for nombre, g in [("Principal sin protocolos especiales", principal_cn),
                      ("CFI-G con yes/no/flash", cfig_cn), ("Flame Design contexto", fd_cn),
                      ("CFI-G ignición positiva y modo de final informado", extincion_cn)]
])
display(cohortes_cn)
print("Las cohortes se superponen: sus tamaños no se suman.")

### Construcción de bloques y resultados descriptivos completos

La tabla de bloques informa qué comparaciones son posibles. Una categoría única
se describe, pero no genera una comparación artificial. `yes/no/flash` y SE/FT
ausentes aparecen con n=0. Faltantes de categoría se conservan en las tablas y
quedan fuera de efectos/figuras comparativas. No se condiciona la comparación
de combustible a que el combustible sea constante.

In [ ]:
especificaciones_cn = {
    "combustible": {"data": principal_cn, "category": "fuel"},
    "dilucion": {"data": principal_cn, "category": "dilution_group"},
    "ignicion_cfig": {"data": cfig_cn, "category": "ignition_outcome", "expected_levels": ORDERS["ignition_outcome"]},
    "extincion_cfig": {"data": extincion_cn, "category": "extinction_mode",
                       "numeric_columns": ["burn_time_s", "peak_burner_t_k"],
                       "expected_levels": ORDERS["extinction_mode"]},
    "derivadas_cfig": {"data": cfig_cn, "category": "ignition_outcome",
                       "numeric_columns": ["po2_kpa", "zst", "tad_k"], "expected_levels": ORDERS["ignition_outcome"]},
    "contexto_fd": {"data": fd_cn, "category": "ignition_outcome",
                    "numeric_columns": ["xo2", "p_kpa", "x_fuel", "m_fuel_mg_s", "m_o2_mg_s", "m_n2_mg_s"],
                    "expected_levels": ORDERS["ignition_outcome"]},
}
resultados_cn = {
    nombre: describe_categories(name=nombre, **especificacion)
    for nombre, especificacion in especificaciones_cn.items()
}
resumen_cn = pd.concat([r.summary for r in resultados_cn.values()], ignore_index=True)
efectos_cn = pd.concat([r.effects for r in resultados_cn.values() if not r.effects.empty], ignore_index=True)
operaciones_cn = pd.concat([r.operation_medians for r in resultados_cn.values()], ignore_index=True)
catalogo_cn = pd.concat([r.panel_catalog for r in resultados_cn.values()], ignore_index=True)
display(catalogo_cn.groupby(["analysis", "status"]).agg(paneles=("panel_id", "size"), bloques=("block_id", "nunique")))
columnas_resumen_cn = ["block_id", "variable", "category_level", "n_total", "n_valid", "n_missing",
                        "n_operations_valid", "median", "q25", "q75", "min", "max",
                        "median_of_operation_medians", "largest_operation_share", "status"]
for nombre, r in resultados_cn.items():
    display(Markdown(f"**{nombre}: resumen completo exportado; primeras 20 filas.**"))
    display(r.summary[columnas_resumen_cn].head(20).round(4))
display(Markdown("**Pares de categorías: diferencia A − B y delta de Cliff, sin inferencia.**"))
columnas_efecto_cn = ["analysis", "block_id", "variable", "category_a", "category_b", "n_a", "n_b",
                       "operations_a", "operations_b", "n_shared_operations", "operations_disjoint",
                       "median_difference_a_minus_b", "cliffs_delta_a_minus_b",
                       "effect_status", "single_operation_category", "univariate_ranges_overlap"]
display(efectos_cn[columnas_efecto_cn].head(40).round(4))

### Distribuciones, pares de categorías y cobertura por operación

Para no fabricar una matriz indiscriminada de gráficos, se preseleccionan variables
por pregunta. Todas las restantes quedan en las tablas: combustible (`xo2`, presión,
fracción de combustible); dilución (fracción y caudal); ignición (`xo2`, presión);
extinción (duración y temperatura); calculadas (`tad_k`); contexto FD (flujo de N₂).

Se guardan todas las figuras de esa selección con dos categorías válidas y al menos
cinco valores en total. Una categoría de menos de cinco valores sigue visible como
puntos, sin boxplot ni efecto. Las figuras mostradas aquí priorizan un primer panel
por bloque y geometría; el manifiesto permite abrir todas las demás.

In [ ]:
variables_figura_cn = {
    "combustible": ["xo2", "p_kpa", "x_fuel"],
    "dilucion": ["x_fuel", "m_fuel_mg_s", "m_o2_mg_s"],
    "ignicion_cfig": ["xo2", "p_kpa"],
    "extincion_cfig": ["burn_time_s", "peak_burner_t_k"],
    "derivadas_cfig": ["tad_k"],
    "contexto_fd": ["m_n2_mg_s"],
}
manifiesto_figuras_cn = []
for nombre, resultado in resultados_cn.items():
    display(Markdown(f"#### {nombre}"))
    seleccion = resultado.panel_catalog.copy()
    seleccion["selected_variable"] = seleccion["variable"].isin(variables_figura_cn[nombre])
    seleccion["render"] = seleccion["selected_variable"] & seleccion["n_levels_valid"].ge(2) & seleccion["n_valid"].ge(5)
    candidatos = seleccion.loc[seleccion["render"]].copy()
    if candidatos.empty:
        print("Sin comparaciones con cobertura suficiente; consultar las tablas de categorías ausentes.")
    # Coverage-based order; never rank by effect size or significance.
    candidatos["variable_priority"] = candidatos["variable"].map({v: i for i, v in enumerate(variables_figura_cn[nombre])})
    candidatos = candidatos.sort_values(["variable_priority", "n_operations", "n_valid", "block_id"],
                                         ascending=[True, False, False, True])
    primeros = candidatos.groupby("flame_type", sort=True).head(1)
    resto = candidatos.loc[~candidatos["panel_id"].isin(primeros["panel_id"])]
    mostrar = set(pd.concat([primeros, resto]).head(MAX_FIGURAS_EN_NOTEBOOK_CN)["panel_id"])
    for _, panel in seleccion.iterrows():
        record = panel.to_dict()
        record["shown_in_notebook"] = panel["panel_id"] in mostrar
        record["png"] = ""
        record["svg"] = ""
        if panel["render"]:
            figura = plot_comparison(resultado, panel["panel_id"])
            stem = panel["panel_id"].replace(":", "__")
            png, svg = FIG_CN / f"{stem}.png", FIG_CN / f"{stem}.svg"
            figura.savefig(png, dpi=120, bbox_inches="tight")
            figura.savefig(svg, bbox_inches="tight")
            record["png"], record["svg"] = str(png.relative_to(OUT_CN)), str(svg.relative_to(OUT_CN))
            if record["shown_in_notebook"]:
                display(figura)
            plt.close(figura)
        record["render_reason"] = (
            "saved" if panel["render"] else "variable_not_selected" if not panel["selected_variable"]
            else "single_or_missing_category" if panel["n_levels_valid"] < 2 else "fewer_than_5_total_values"
        )
        manifiesto_figuras_cn.append(record)
    print(f"Figuras guardadas: {len(candidatos)}; mostradas: {len(mostrar)}.")
manifiesto_figuras_cn = pd.DataFrame(manifiesto_figuras_cn)

### Procedencia, protocolos y variables con interpretación limitada

La composición de los grupos ayuda a reconocer confusión entre combustible, periodo
y campaña. Estos paneles no estiman un efecto de campaña ni comparan normal/inversa
como si las composiciones describieran el mismo lugar físico. Las frecuencias de
`cool_flame` y `tfp` incluyen explícitamente las respuestas ausentes.

In [ ]:
factores_procedencia_cn = ["campaign", "flame_type", "period_group", "protocol_group", "outcome_source"]
composicion_cn = (
    gas_unicos.groupby(factores_procedencia_cn + ["fuel", "dilution_group"], dropna=False)
    .agg(n_pruebas=("test_id", "size"), n_operaciones=("operation_group", "nunique"),
         sin_timestamp=("timestamp_verified", lambda s: int((~s).sum())),
         n_anomalias=("audit_ok", lambda s: int((~s).sum())))
    .reset_index()
)
variables_procedencia_cn = ["xo2", "p_kpa", "x_fuel", "m_fuel_mg_s", "m_o2_mg_s", "m_n2_mg_s"]
cobertura_procedencia_cn = (
    gas_unicos.groupby(factores_procedencia_cn, dropna=False)[variables_procedencia_cn]
    .agg(lambda s: 100 * s.replace([np.inf, -np.inf], np.nan).isna().mean()).astype(float)
    .reset_index()
)
display(composicion_cn)
fig, ax = plt.subplots(figsize=(11, max(4, len(cobertura_procedencia_cn) * 0.30)))
matriz_procedencia_cn = cobertura_procedencia_cn.set_index(factores_procedencia_cn)
sns.heatmap(matriz_procedencia_cn, annot=True, fmt=".0f", vmin=0, vmax=100, cmap="YlOrRd",
            cbar_kws={"label": "% ausentes"}, ax=ax)
ax.set(title="Cobertura según procedencia y protocolo", ylabel="Campaña | geometría | periodo | protocolo | fuente")
plt.xticks(rotation=35, ha="right")
fig.savefig(FIG_CN / "procedencia_faltantes.png", dpi=120, bbox_inches="tight")
fig.savefig(FIG_CN / "procedencia_faltantes.svg", bbox_inches="tight")
plt.show()
plt.close(fig)

# All nitrogen-feed observations stay in their original source/protocol blocks.
nitrogeno_cn = (
    gas_unicos.loc[gas_unicos["campaign"].eq("Flame Design")]
    .groupby(factores_procedencia_cn, dropna=False)
    .agg(n_total=("test_id", "size"), n_valid=("m_n2_mg_s", "count"),
         n_operaciones=("operation_group", "nunique"),
         mediana_mg_s=("m_n2_mg_s", "median"), minimo_mg_s=("m_n2_mg_s", "min"), maximo_mg_s=("m_n2_mg_s", "max"))
    .reset_index()
)
display(Markdown("**Nitrógeno suministrado: resumen exclusivo de Flame Design.**"))
display(nitrogeno_cn)
frecuencias_flags_cn = []
for variable in ["cool_flame", "tfp"]:
    for (campana, tipo), g in gas_unicos.groupby(["campaign", "flame_type"]):
        niveles = g[variable].astype("string").fillna("sin_respuesta")
        for nivel, n in niveles.value_counts().items():
            sub = g.loc[niveles.eq(nivel)]
            frecuencias_flags_cn.append({"variable": variable, "campaign": campana, "flame_type": tipo,
                                         "category_level": nivel, "n": int(n),
                                         "n_operations": sub["operation_group"].nunique(),
                                         "pct_grupo": 100 * n / len(g)})
frecuencias_flags_cn = pd.DataFrame(frecuencias_flags_cn)
display(frecuencias_flags_cn.round(2))
columnas_flags_cn = ["psi_id", "test_id", "family_code", "operation_group", "cool_flame", "tfp",
                      "ignition_outcome", "xo2", "p_kpa", "source_file", "source_row"]
registros_flags_cn = gas_unicos.loc[gas_unicos["cool_flame"].notna() | gas_unicos["tfp"].notna(), columnas_flags_cn]
display(Markdown("**Todas las respuestas informadas de cool_flame:**"))
display(registros_flags_cn.loc[registros_flags_cn["cool_flame"].notna()])
print("Los registros TFP completos se exportan; su acrónimo sigue pendiente de definición.")

### Dependencia y análisis de sensibilidad

Los rombos y las tablas por operación conservan las réplicas. La diferencia entre
mediana de pruebas y mediana de medianas señala sensibilidad a la ponderación; no
es una prueba estadística. La fracción de filas de la operación dominante se reporta
para cada categoría. Sin fecha verificable y con cronología anómala son problemas
distintos y se evalúan por separado.

Las sensibilidades no renombran protocolos para mezclarlos: los controles y rampas
conservan sus bloques y se describen como cohortes aparte. Al excluir fechas no
verificadas se mantienen también las categorías y bloques que desaparecen, mediante
un enlace externo de resúmenes que deja explícita su ausencia.

In [ ]:
escenarios_cn = {
    "principal": principal_cn,
    "solo_timestamp_verificado": principal_cn.loc[principal_cn["timestamp_verified"]].copy(),
    "incluir_cronologia_anomala": gas_unicos.loc[
        gas_unicos["protocol_group"].eq("sin_anotacion")
        & ~gas_unicos["event_identity_conflict"] & ~gas_unicos["event_op_mismatch"]
        & ~gas_unicos["temporal_collision"]
    ].copy(),
}
sensibilidad_cn = []
for escenario, fuente in escenarios_cn.items():
    for nombre in ["combustible", "dilucion", "ignicion_cfig"]:
        fuente_actual = fuente.loc[fuente["campaign"].eq("CFI-G")] if nombre == "ignicion_cfig" else fuente
        spec = {k: v for k, v in especificaciones_cn[nombre].items() if k != "data"}
        resultado = resultados_cn[nombre] if escenario == "principal" else describe_categories(
            fuente_actual, name=nombre, scenario=escenario, **spec
        )
        sensibilidad_cn.append(resultado.summary)
sensibilidad_cn = pd.concat(sensibilidad_cn, ignore_index=True)
llaves_sensibilidad_cn = ["analysis", "block_label", "category", "category_level", "variable"]
referencia_cn = sensibilidad_cn.loc[sensibilidad_cn["scenario"].eq("principal"),
    llaves_sensibilidad_cn + ["n_total", "n_valid", "n_operations_valid", "median", "median_of_operation_medians"]]
cambios_sensibilidad_cn = []
for escenario in ["solo_timestamp_verificado", "incluir_cronologia_anomala"]:
    actual = sensibilidad_cn.loc[sensibilidad_cn["scenario"].eq(escenario),
        llaves_sensibilidad_cn + ["n_total", "n_valid", "n_operations_valid", "median", "median_of_operation_medians"]]
    cambio = referencia_cn.merge(actual, on=llaves_sensibilidad_cn, how="outer",
                                suffixes=("_principal", "_sensibilidad"), validate="one_to_one", indicator=True)
    cambio["scenario"] = escenario
    cambio["median_change"] = cambio["median_sensibilidad"] - cambio["median_principal"]
    cambios_sensibilidad_cn.append(cambio)
cambios_sensibilidad_cn = pd.concat(cambios_sensibilidad_cn, ignore_index=True)
diferencias_ponderacion_cn = resumen_cn.copy()
diferencias_ponderacion_cn["median_difference_weighting"] = (
    diferencias_ponderacion_cn["median"] - diferencias_ponderacion_cn["median_of_operation_medians"]
)
display(diferencias_ponderacion_cn.loc[
    diferencias_ponderacion_cn["n_operations_valid"].le(1) | diferencias_ponderacion_cn["largest_operation_share"].gt(0.5),
    ["analysis", "block_id", "variable", "category_level", "n_valid", "n_operations_valid", "largest_operation_share"]
].head(30).round(3))
mostrar_cambios_cn = cambios_sensibilidad_cn.loc[
    cambios_sensibilidad_cn["_merge"].ne("both")
    | cambios_sensibilidad_cn["n_valid_principal"].ne(cambios_sensibilidad_cn["n_valid_sensibilidad"])
]
display(mostrar_cambios_cn[["analysis", "scenario", "category_level", "variable", "n_valid_principal",
                           "n_valid_sensibilidad", "median_principal", "median_sensibilidad", "_merge"]].head(40))
protocolos_revision_cn = gas_unicos.loc[
    ~gas_unicos["protocol_group"].eq("sin_anotacion") | ~gas_unicos["audit_ok"]
].copy()
resumen_protocolos_revision_cn = (
    protocolos_revision_cn.groupby(["campaign", "flame_type", "protocol_group", "audit_ok"], dropna=False)
    .agg(n=("test_id", "size"), operaciones=("operation_group", "nunique"),
         xo2_validos=("xo2", "count"), presion_validos=("p_kpa", "count"),
         x_fuel_validos=("x_fuel", "count"),
         etiquetas_conocidas=("ignition_outcome", "count"))
    .reset_index()
)
display(resumen_protocolos_revision_cn)

### Hallazgos descriptivos y límites antes del modelado

Las conclusiones siguientes se calculan con la ejecución actual. No se ordenan
variables por «significancia» ni se presentan diferencias de condiciones como efectos
de cambiar el combustible. Se conservan las tablas completas para revisar cada bloque.

In [ ]:
tablas_cn = {
    "fuentes": fuentes_cn, "cohortes": cohortes_cn, "resumen_categorias": resumen_cn,
    "efectos_descriptivos": efectos_cn, "medianas_operacion": operaciones_cn,
    "catalogo_comparaciones": catalogo_cn, "manifiesto_figuras": manifiesto_figuras_cn,
    "composicion_procedencia": composicion_cn, "cobertura_procedencia": cobertura_procedencia_cn,
    "nitrogeno_fd": nitrogeno_cn, "frecuencias_flags": frecuencias_flags_cn,
    "registros_flags": registros_flags_cn, "sensibilidad_resumen": sensibilidad_cn,
    "sensibilidad_cambios": cambios_sensibilidad_cn, "ponderacion_operaciones": diferencias_ponderacion_cn,
    "protocolos_revision": protocolos_revision_cn, "protocolos_revision_resumen": resumen_protocolos_revision_cn,
}
for nombre, tabla in tablas_cn.items():
    tabla.to_csv(OUT_CN / f"{nombre}.csv", index=False, encoding="utf-8-sig")
# Export point-level provenance for every panel, retaining trial identity.
puntos_cn = pd.concat([
    puntos.assign(panel_id=panel_id, analysis=nombre)
    for nombre, resultado in resultados_cn.items() for panel_id, puntos in resultado.panels.items()
], ignore_index=True)
puntos_cn.to_csv(OUT_CN / "observaciones_por_panel.csv", index=False, encoding="utf-8-sig")
assert not puntos_cn.duplicated(["panel_id", "psi_id", "test_id"]).any(), "Filas multiplicadas dentro de un panel"
for p, esperado in zip(entradas_cn, fuentes_cn["sha256"], strict=True):
    assert hashlib.sha256(p.read_bytes()).hexdigest() == esperado, "Cambió una fuente durante el análisis"
efectos_validos_cn = efectos_cn.loc[efectos_cn["effect_status"].eq("descriptive_only")]
flags_poca_info_cn = resumen_cn.loc[resumen_cn["n_operations_valid"].le(1) & resumen_cn["n_valid"].gt(0)]
n_flash_cn = int(cfig_cn["ignition_outcome"].eq("flash").sum())
combustibles_flash_cn = ", ".join(sorted(cfig_cn.loc[cfig_cn["ignition_outcome"].eq("flash"), "fuel"].unique()))
reporte_cn = {
    "protocol": "descriptive_only", "inputs": fuentes_cn.to_dict(orient="records"),
    "n_source_trials": len(gas_unicos), "n_principal_trials": len(principal_cn),
    "n_summary_rows": len(resumen_cn), "n_pairwise_rows": len(efectos_cn),
    "n_descriptive_effects": len(efectos_validos_cn),
    "n_saved_comparison_figures": int(manifiesto_figuras_cn["render"].sum()),
    "n_shown_comparison_figures": int(manifiesto_figuras_cn["shown_in_notebook"].sum()),
    "pvalues_computed": False, "confidence_intervals_computed": False, "models_fitted": False,
    "imputation": False, "replicas_removed": False,
    "effect_orientation": "A minus B; Cliff delta = P(A>B)-P(A<B)",
    "thresholds": {"min_effect_per_category": 5, "min_boxplot": 5,
                   "min_violin": 20, "min_violin_unique": 5},
}
(OUT_CN / "resumen.json").write_text(json.dumps(reporte_cn, ensure_ascii=False, indent=2), encoding="utf-8")
display(Markdown(f"""
**Resultado de esta ejecución**

- La cohorte principal reúne **{len(principal_cn)} pruebas**, separadas en bloques;
  no representa una muestra aleatoria de atmósferas de misión.
- Se exportaron **{len(resumen_cn)} resúmenes categoría–variable–bloque** y
  **{len(efectos_validos_cn)} efectos descriptivos con al menos cinco valores por categoría**.
  Los pares restantes conservan el motivo por el que no se calcula el efecto.
- **{len(flags_poca_info_cn)} celdas observadas de resumen** dependen de una sola operación.
  Una diferencia grande en esas celdas no demuestra reproducibilidad entre jornadas.
- Los **{n_flash_cn} flash** observados en esta cohorte corresponden a **{combustibles_flash_cn}**.
  Su distribución de condiciones no permite atribuir el fenómeno exclusivamente al combustible.
- Combustible, dilución, presión y composición pertenecen a un diseño experimental conjunto.
  Las diferencias observadas y el solapamiento de rangos no resuelven la confusión causal.
  En **{int(efectos_cn['operations_disjoint'].sum())} contrastes variable–par de categorías**
  no hay operaciones compartidas: no puede separarse el efecto de categoría del de jornada
  a partir de esa comparación descriptiva por sí sola.
- `m_n2_mg_s` permanece como alimentación de Flame Design; `cool_flame` tiene cobertura
  muy limitada y `tfp` conserva una semántica pendiente. No se rellenan sus ausencias.
- Los tiempos de final por corte de flujo no se interpretan como tiempos de extinción natural.
  No se calculan p-valores, intervalos ni modelos en esta sección.

**Antes de modelar:** decidir la etiqueta `flash`, revisar equivalencia de resultados entre
campañas, resolver la cronología anómala y comprobar soporte común y operaciones disponibles
para cada pregunta. Una etapa posterior requerirá validación agrupada y definición del objetivo.

Resultados y figuras: `{OUT_CN}`. Objetos principales: `resultados_cn`, `resumen_cn`,
`efectos_cn`, `operaciones_cn`, `cambios_sensibilidad_cn` y `manifiesto_figuras_cn`.
"""))

# A pre-specified scientific question from the project: flash versus yes/no in ethane.
foco_flash_cn = resumen_cn.loc[
    resumen_cn["analysis"].eq("ignicion_cfig") & resumen_cn["flame_type"].eq("inverse")
    & resumen_cn["fuel"].eq("ethane") & resumen_cn["period_group"].eq("2021")
    & resumen_cn["variable"].eq("xo2"),
    ["block_id", "category_level", "n_valid", "n_operations_valid", "median", "q25", "q75", "min", "max"],
]
display(Markdown("**Foco predefinido: oxígeno en CFI-G inversa con etano, por resultado.**"))
display(foco_flash_cn.round(4))
display(Markdown(
    "Estas distribuciones son condicionales a la familia y al diseño observado. "
    "La categoría `no` contiene pocas observaciones; una diferencia de medianas no establece "
    "un umbral de ignición, y flash sigue sin etiqueta binaria."
))
# <!-- FLAMESAFE:CATEGORICAL:END -->